# Customer Churn Prediction – Classification & Business Analytics
**Author:** Kishan Sheladiya  
**Stack:** Python · Pandas · Scikit-learn · XGBoost · imbalanced-learn (SMOTE) · Matplotlib · Seaborn

---
This notebook covers:
1. Data exploration & churn analysis
2. Feature distributions by churn label
3. Preprocessing: encoding, scaling, SMOTE
4. Model training: Logistic Regression, Random Forest, XGBoost
5. ROC curves & confusion matrices
6. Business recommendations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/sample/customer_data.csv')
print(f'Shape: {df.shape}')
print(f'Churn rate: {df.churn.mean():.1%}')
df.head()

## 1. Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], counts.values, color=['#16a34a', '#ef4444'])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')
axes[0].set_title('Churn Distribution'); axes[0].set_ylabel('Count')

df['churn'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#16a34a','#ef4444'],
                                     labels=['No Churn','Churn'], ax=axes[1])
axes[1].set_title('Churn Rate'); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

## 2. Key Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Tenure
for label, color, name in [(0,'#16a34a','No Churn'),(1,'#ef4444','Churn')]:
    axes[0,0].hist(df[df.churn==label]['tenure'], bins=30, alpha=0.6, color=color, label=name)
axes[0,0].set_title('Tenure Distribution'); axes[0,0].legend()

# Monthly charges
for label, color, name in [(0,'#16a34a','No Churn'),(1,'#ef4444','Churn')]:
    axes[0,1].hist(df[df.churn==label]['monthly_charges'], bins=30, alpha=0.6, color=color, label=name)
axes[0,1].set_title('Monthly Charges Distribution'); axes[0,1].legend()

# Contract type
contract_churn = df.groupby('contract')['churn'].mean().sort_values(ascending=False)
axes[1,0].bar(contract_churn.index, contract_churn.values*100, color=['#ef4444','#f97316','#16a34a'])
axes[1,0].set_title('Churn Rate by Contract Type'); axes[1,0].set_ylabel('Churn Rate (%)')

# Internet service
internet_churn = df.groupby('internet_service')['churn'].mean().sort_values(ascending=False)
axes[1,1].bar(internet_churn.index, internet_churn.values*100, color=['#ef4444','#f97316','#16a34a'])
axes[1,1].set_title('Churn Rate by Internet Service'); axes[1,1].set_ylabel('Churn Rate (%)')

plt.suptitle('Churn Analysis by Key Features', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Correlation Heatmap

In [ ]:
num_cols = ['tenure','age','num_products','monthly_charges','total_charges',
            'has_partner','has_dependents','online_security','tech_support',
            'paperless_billing','churn']
plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

## 4. Preprocessing & SMOTE

In [ ]:
import sys
sys.path.insert(0, '../src')
from preprocess import preprocess

X_train, X_test, y_train, y_test, scaler, feature_cols = preprocess('../data/sample/customer_data.csv')
print(f'Features: {feature_cols}')

## 5. Model Training & Results

In [ ]:
import json, os

results = {}
for fname in ['logistic_regression', 'random_forest', 'xgboost']:
    fpath = f'../results/metrics/{fname}.json'
    if os.path.exists(fpath):
        with open(fpath) as f:
            d = json.load(f)
        results[fname.replace('_',' ').title()] = {
            'ROC-AUC': round(d['roc_auc'], 3),
            'F1':      round(d['f1'], 3),
        }

comp = pd.DataFrame(results).T
print(comp)
comp

In [ ]:
from IPython.display import Image, display

for plot in ['roc_curves', 'feature_importance', 'cm_random_forest']:
    fpath = f'../results/plots/{plot}.png'
    if os.path.exists(fpath):
        display(Image(filename=fpath))

## 6. Business Recommendations

Based on the model analysis, the highest-risk customers share these characteristics:

- **Month-to-month contracts** → highest churn rate; offer incentives to switch to annual plans
- **High monthly charges + short tenure** → price-sensitive new customers; consider welcome discounts
- **Fiber optic without online security/tech support** → upsell bundled support packages
- **Electronic check payment** → correlates with churn; encourage auto-payment setup

A churn probability score above 0.5 flags customers for proactive retention outreach.
